# Instruction finetuning GPT-2 small (124M) from scratch

Raw PyTorch, in the style of *Build a Large Language Model (From Scratch)* by Sebastian Raschka (ch. 7).

`transformers` is used **only** to download the pretrained weights and the BPE tokenizer.
The architecture, the dataset, the padding/masking collate function, the loss, the optimization
loop and the sampling are all written out by hand — no `Trainer`, no `datasets`, no `peft`.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers matplotlib

## 1. Setup

In [ ]:
import json
import time
from functools import partial
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(123)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

PAD_TOKEN_ID = 50256   # <|endoftext|> — GPT-2 has no dedicated padding token
IGNORE_INDEX = -100    # cross_entropy skips positions with this label

print("torch:", torch.__version__)
print("device:", device)

## 2. Data

Upload `instruction-data-with-response.json` when the file picker appears
(or mount Drive and point `DATA_FILE` at it).

Each record has `instruction`, `input`, `output` and `model_response`. Only the first three are
used for training — `model_response` came from an earlier model and is ignored here.

In [ ]:
DATA_FILE = "instruction-data-with-response.json"

if not Path(DATA_FILE).exists():
    from google.colab import files
    print(f"Upload {DATA_FILE}:")
    uploaded = files.upload()
    DATA_FILE = next(iter(uploaded))

with open(DATA_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of entries:", len(data))
print("With an input field:", sum(1 for e in data if e["input"]))
data[0]

## 3. Prompt formatting (Alpaca style)

The model only ever sees one long string. We split it into a *prompt* part and a *response* part
so that later we can optionally mask the prompt out of the loss.

In [ ]:
def format_input(entry):
    """Turn one JSON record into the prompt half of the training example."""
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input") else ""
    return instruction_text + input_text


def format_response(entry):
    return f"\n\n### Response:\n{entry['output']}"


print(format_input(data[0]) + format_response(data[0]))
print("\n" + "=" * 70 + "\n")
print(format_input(data[1]) + format_response(data[1]))

In [ ]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

## 4. Dataset and collate function

The interesting part. `custom_collate_fn` does three things per batch:

1. appends one `<|endoftext|>` to every sequence (so the model learns to stop) and pads to the
   longest sequence *in that batch* — not to a fixed 1024, which would waste compute;
2. builds `targets` as `inputs` shifted left by one position;
3. replaces padding in `targets` with `-100` so it contributes no loss — except the **first**
   pad token, which is the end-of-text the model should actually predict.

`mask_prompt=True` additionally removes the instruction tokens from the loss, so only the
response is trained on (the book's optional exercise).

In [ ]:
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
print(tokenizer.encode("Hello, world!"))
print(tokenizer.decode([50256]))

In [ ]:
class InstructionDataset(Dataset):
    """Pre-tokenizes every `prompt + response` pair once, up front."""

    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        self.prompt_lengths = []   # only needed when masking the prompt

        for entry in data:
            prompt = format_input(entry)
            full_text = prompt + format_response(entry)
            self.encoded_texts.append(tokenizer.encode(full_text))
            self.prompt_lengths.append(len(tokenizer.encode(prompt)))

    def __getitem__(self, index):
        return {
            "ids": self.encoded_texts[index],
            "prompt_len": self.prompt_lengths[index],
        }

    def __len__(self):
        return len(self.data)

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=PAD_TOKEN_ID,
    ignore_index=IGNORE_INDEX,
    allowed_max_length=None,
    mask_prompt=False,
    device="cpu",
):
    batch_max_length = max(len(item["ids"]) + 1 for item in batch)

    inputs_lst, targets_lst = [], []
    for item in batch:
        new_item = item["ids"] + [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # Keep the first <|endoftext|> as a real target, ignore the rest.
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # Optional: don't compute a loss on the instruction itself.
        # targets[i] predicts padded[i + 1], so the response starts at i = prompt_len - 1.
        if mask_prompt:
            targets[: item["prompt_len"] - 1] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    return torch.stack(inputs_lst).to(device), torch.stack(targets_lst).to(device)

In [ ]:
# Sanity check on a toy batch: targets are inputs shifted by one, padding is -100.
toy = [{"ids": [1, 2, 3, 4, 5], "prompt_len": 2}, {"ids": [6, 7], "prompt_len": 1}]
ti, tt = custom_collate_fn(toy)
print("inputs\n", ti)
print("targets\n", tt)

ti, tt = custom_collate_fn(toy, mask_prompt=True)
print("targets with mask_prompt=True\n", tt)

In [ ]:
BATCH_SIZE = 8
CONTEXT_LENGTH = 1024
MASK_PROMPT = False   # True = train on response tokens only

collate = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=CONTEXT_LENGTH,
    mask_prompt=MASK_PROMPT,
)

loader_kwargs = dict(batch_size=BATCH_SIZE, collate_fn=collate, num_workers=0)

train_dataset = InstructionDataset(train_data, tokenizer)
val_dataset = InstructionDataset(val_data, tokenizer)
test_dataset = InstructionDataset(test_data, tokenizer)

train_loader = DataLoader(train_dataset, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)

for inputs, targets in train_loader:
    print("inputs:", inputs.shape, "targets:", targets.shape)
    break
print(f"batches per epoch: {len(train_loader)}  "
      f"(val {len(val_loader)}, test {len(test_loader)})")

### Token statistics

Worth checking before training starts: whether anything is truncated at 1024 tokens, how much of
each padded batch is real content rather than ignored padding, and what share of the sequence is
prompt rather than response — that last number is exactly what `MASK_PROMPT=True` removes from
the loss.

In [ ]:
lengths = [len(ids) for ids in train_dataset.encoded_texts]
prompt_lens = train_dataset.prompt_lengths
sorted_lengths = sorted(lengths)

print(f"Tokens per example: min {min(lengths)}, "
      f"median {sorted_lengths[len(lengths) // 2]}, "
      f"mean {sum(lengths) / len(lengths):.1f}, max {max(lengths)}")
print(f"Truncated at the {CONTEXT_LENGTH}-token context: "
      f"{sum(1 for n in lengths if n > CONTEXT_LENGTH)} of {len(lengths)}")
print(f"Prompt share of the sequence: {sum(prompt_lens) / sum(lengths):.1%} "
      f"(what MASK_PROMPT=True would drop from the loss)")

# How much of each padded batch is a real target vs. an ignored pad position?
real = total = 0
for _, targets in train_loader:
    total += targets.numel()
    real += (targets != IGNORE_INDEX).sum().item()
print(f"Padding overhead: {1 - real / total:.1%} of target positions are ignored")
print(f"Trainable tokens per epoch: {real:,}")

plt.figure(figsize=(5, 3))
plt.hist(lengths, bins=20)
plt.xlabel("Tokens per training example")
plt.ylabel("Count")
plt.title("Sequence length distribution")
plt.tight_layout()
plt.show()

## 5. GPT-2 written from scratch

Twelve pre-LayerNorm transformer blocks, causal multi-head attention, tanh-approximated GELU.

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,       # BPE vocabulary size
    "context_length": 1024,    # maximum number of positions
    "emb_dim": 768,            # embedding / residual stream width
    "n_heads": 12,             # attention heads per block
    "n_layers": 12,            # transformer blocks
    "drop_rate": 0.0,          # 0.0 while finetuning
    "qkv_bias": True,          # OpenAI's GPT-2 uses biases in the qkv projection
}

In [ ]:
class MultiHeadAttention(nn.Module):
    """Causal self-attention with all heads computed in one batched matmul."""

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)   # mixes the heads back together
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool(),
            persistent=False,
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # (b, num_tokens, d_out) -> (b, num_heads, num_tokens, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(
            self.mask[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm written out so the normalization statistics stay visible."""

    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    """The tanh approximation of GELU that GPT-2 was trained with."""

    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi))
                * (x + 0.044715 * torch.pow(x, 3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-LayerNorm block: x + attn(ln(x)), then x + ff(ln(x))."""

    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        return x + shortcut

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)   # logits, (b, seq_len, vocab_size)

## 6. Load the pretrained OpenAI weights

The checkpoint stores each projection as a `Conv1D` matrix of shape `(in, out)`, while `nn.Linear`
wants `(out, in)` — and newer `transformers` releases already store some of them transposed. So we
pick the orientation by shape rather than by library version. The fused `c_attn` matrix holds
query, key and value stacked together and gets split into three.

In [ ]:
def _as_linear_weight(w, out_features, in_features):
    if tuple(w.shape) == (in_features, out_features):
        return w.T.contiguous()
    assert tuple(w.shape) == (out_features, in_features), f"unexpected shape {tuple(w.shape)}"
    return w.contiguous()


def load_weights_from_hf(model, hf_model):
    """Copy the pretrained OpenAI GPT-2 weights into our GPTModel."""
    sd = hf_model.state_dict()
    emb = model.pos_emb.weight.shape[1]

    with torch.no_grad():
        model.tok_emb.weight.copy_(sd["transformer.wte.weight"])
        model.pos_emb.weight.copy_(sd["transformer.wpe.weight"])

        for i, block in enumerate(model.trf_blocks):
            p = f"transformer.h.{i}."

            # The qkv projection is one fused matrix in the checkpoint.
            w_qkv = _as_linear_weight(sd[p + "attn.c_attn.weight"], 3 * emb, emb)
            b_qkv = sd[p + "attn.c_attn.bias"]
            q_w, k_w, v_w = torch.split(w_qkv, emb, dim=0)
            q_b, k_b, v_b = torch.split(b_qkv, emb, dim=0)
            block.att.W_query.weight.copy_(q_w)
            block.att.W_key.weight.copy_(k_w)
            block.att.W_value.weight.copy_(v_w)
            block.att.W_query.bias.copy_(q_b)
            block.att.W_key.bias.copy_(k_b)
            block.att.W_value.bias.copy_(v_b)

            block.att.out_proj.weight.copy_(
                _as_linear_weight(sd[p + "attn.c_proj.weight"], emb, emb)
            )
            block.att.out_proj.bias.copy_(sd[p + "attn.c_proj.bias"])

            block.ff.layers[0].weight.copy_(
                _as_linear_weight(sd[p + "mlp.c_fc.weight"], 4 * emb, emb)
            )
            block.ff.layers[0].bias.copy_(sd[p + "mlp.c_fc.bias"])
            block.ff.layers[2].weight.copy_(
                _as_linear_weight(sd[p + "mlp.c_proj.weight"], emb, 4 * emb)
            )
            block.ff.layers[2].bias.copy_(sd[p + "mlp.c_proj.bias"])

            block.norm1.scale.copy_(sd[p + "ln_1.weight"])
            block.norm1.shift.copy_(sd[p + "ln_1.bias"])
            block.norm2.scale.copy_(sd[p + "ln_2.weight"])
            block.norm2.shift.copy_(sd[p + "ln_2.bias"])

        model.final_norm.scale.copy_(sd["transformer.ln_f.weight"])
        model.final_norm.shift.copy_(sd["transformer.ln_f.bias"])
        # GPT-2 ties the output head to the token embedding matrix.
        model.out_head.weight.copy_(sd["transformer.wte.weight"])

    return model

In [ ]:
from transformers import GPT2LMHeadModel

hf_model = GPT2LMHeadModel.from_pretrained("gpt2").eval()

model = GPTModel(GPT_CONFIG_124M)
load_weights_from_hf(model, hf_model)
model.eval()

# Verify our implementation reproduces the reference model, then drop it.
probe = torch.tensor([tokenizer.encode("The capital of France is Paris, and the capital of Germany is")])
with torch.no_grad():
    ref_logits = hf_model(probe).logits
    our_logits = model(probe)

print("max abs logit difference:", (ref_logits - our_logits).abs().max().item())
print("same argmax everywhere:", (ref_logits.argmax(-1) == our_logits.argmax(-1)).all().item())

del hf_model
model.to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 7. Sampling loop

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    """Autoregressive sampling, written out token by token."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]   # never feed more than the context window
        logits = model(idx_cond)
        logits = logits[:, -1, :]           # only the last position predicts the next token

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits
            )

        if temperature > 0.0:
            probs = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and (idx_next == eos_id).all():
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx


def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0)   # add batch dimension


def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

In [ ]:
# What the base model does before finetuning: it rambles instead of answering.
prompt = format_input(val_data[0]) + "\n\n### Response:\n"
token_ids = generate(
    model, text_to_token_ids(prompt, tokenizer).to(device),
    max_new_tokens=60, context_size=CONTEXT_LENGTH, eos_id=PAD_TOKEN_ID,
)
print(prompt)
print(">>>", token_ids_to_text(token_ids, tokenizer)[len(prompt):])

## 8. Loss

Logits come out as `(batch, tokens, vocab)`; flattening the first two axes turns next-token
prediction into an ordinary classification problem over the vocabulary. Positions labelled `-100`
are skipped by `cross_entropy` automatically.

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    num_batches = min(num_batches or len(data_loader), len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        with torch.no_grad():
            total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
    return total_loss / num_batches


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
# Baseline over the *full* loaders, so it is directly comparable to the numbers after training.
with torch.no_grad():
    base_train = calc_loss_loader(train_loader, model, device)
    base_val = calc_loss_loader(val_loader, model, device)
    base_test = calc_loss_loader(test_loader, model, device)

print(f"Before finetuning -> train {base_train:.3f}, "
      f"val {base_val:.3f}, test {base_test:.3f}")
print(f"Validation perplexity: {torch.exp(torch.tensor(base_val)):.1f}")

## 9. Training loop

The classic four lines: zero the gradients, forward + loss, backward, step.

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    token_ids = generate(model, encoded, max_new_tokens=50,
                         context_size=context_size, eos_id=PAD_TOKEN_ID)
    decoded = token_ids_to_text(token_ids, tokenizer)
    print(decoded[len(start_context):].replace("\n", " "))
    model.train()


def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    step_losses = []   # the raw per-batch loss, free to record and makes a denser curve
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()                                    # reset gradients
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()                                          # backprop
            optimizer.step()                                         # update weights
            tokens_seen += input_batch.numel()
            global_step += 1
            step_losses.append(loss.item())

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch + 1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen, step_losses

In [ ]:
NUM_EPOCHS = 2
LEARNING_RATE = 5e-5
EVAL_FREQ = 2   # only ~11 steps per epoch here, so evaluate often enough to draw a curve

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)
start_context = format_input(val_data[0]) + "\n\n### Response:\n"

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()
train_losses, val_losses, tokens_seen, step_losses = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=NUM_EPOCHS, eval_freq=EVAL_FREQ, eval_iter=5,
    start_context=start_context, tokenizer=tokenizer,
)
elapsed = time.time() - start_time

print(f"\nTraining completed in {elapsed / 60:.2f} minutes")
print(f"Optimizer steps: {len(step_losses)}  "
      f"({elapsed / len(step_losses):.2f} s/step)")
print(f"Tokens seen: {tokens_seen[-1]:,}  ({tokens_seen[-1] / elapsed:,.0f} tokens/s)")
if device.type == "cuda":
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## 10. Loss curves

In [ ]:
from matplotlib.ticker import MaxNLocator

epochs_seen = torch.linspace(0, NUM_EPOCHS, len(train_losses))
steps_seen = torch.linspace(0, NUM_EPOCHS, len(step_losses))

fig, ax1 = plt.subplots(figsize=(6, 3.5))

# Raw per-batch loss in the background: noisy, but shows what the optimizer actually saw.
ax1.plot(steps_seen, step_losses, color="tab:blue", alpha=0.25, linewidth=1,
         label="Per-batch training loss")
ax1.plot(epochs_seen, train_losses, color="tab:blue", label="Training loss")
ax1.plot(epochs_seen, val_losses, color="tab:orange", linestyle="-.", label="Validation loss")
ax1.axhline(base_val, color="gray", linestyle=":", linewidth=1,
            label="Validation loss before finetuning")

ax1.set_xlabel("Epochs")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper right", fontsize=8)
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

ax2 = ax1.twiny()   # second x-axis showing tokens processed
ax2.plot(tokens_seen, train_losses, alpha=0)
ax2.set_xlabel("Tokens seen")

fig.tight_layout()
plt.savefig("loss-plot.pdf")
plt.show()

## 11. Before vs. after

The curve above is measured on 5 sampled batches per evaluation. These numbers use the **full**
loaders, and include the test split, which the training loop never looks at.

Perplexity is just `exp(loss)` — the effective number of tokens the model is choosing between at
each position. It's easier to read than a cross-entropy value.

A caveat worth remembering: 93 training examples is tiny, so a falling training loss with a flat
or rising validation loss is memorization, not learning. That is what the gap between the two
curves tells you.

In [ ]:
model.eval()
with torch.no_grad():
    final_train = calc_loss_loader(train_loader, model, device)
    final_val = calc_loss_loader(val_loader, model, device)
    final_test = calc_loss_loader(test_loader, model, device)

rows = [
    ("train", base_train, final_train, len(train_data)),
    ("val", base_val, final_val, len(val_data)),
    ("test", base_test, final_test, len(test_data)),
]

print(f"{'split':<8}{'n':>5}{'loss before':>14}{'loss after':>13}"
      f"{'change':>10}{'ppl before':>13}{'ppl after':>12}")
print("-" * 75)
for name, before, after, n in rows:
    ppl_before = torch.exp(torch.tensor(before)).item()
    ppl_after = torch.exp(torch.tensor(after)).item()
    print(f"{name:<8}{n:>5}{before:>14.3f}{after:>13.3f}"
          f"{after - before:>+10.3f}{ppl_before:>13.1f}{ppl_after:>12.1f}")

gap = final_val - final_train
print(f"\nGeneralization gap (val - train): {gap:+.3f}")
print("Overfitting: the model is memorizing the training set."
      if gap > 0.5 else "Train and validation loss are still close together.")

## 12. Responses on the held-out test set

Greedy decoding, stopping at `<|endoftext|>` — the token the collate function taught the model
to emit at the end of a response.

Loss is not the thing you actually care about here, so the final cell also reports how often the
model stopped on its own rather than running to the token limit. A model that never emits
`<|endoftext|>` will babble past the end of every answer.

In [ ]:
MAX_NEW_TOKENS = 256

response_lengths, stopped_on_own = [], 0

for entry in test_data:
    prompt = format_input(entry) + "\n\n### Response:\n"
    prompt_ids = text_to_token_ids(prompt, tokenizer).to(device)
    token_ids = generate(
        model,
        prompt_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        context_size=CONTEXT_LENGTH,
        eos_id=PAD_TOKEN_ID,
    )
    generated = token_ids_to_text(token_ids, tokenizer)[len(prompt):]
    entry["gpt2_finetuned_response"] = generated.strip()

    n_new = token_ids.shape[1] - prompt_ids.shape[1]
    response_lengths.append(n_new)
    stopped_on_own += int(n_new < MAX_NEW_TOKENS)   # generate() broke on <|endoftext|>

    print("-" * 70)
    print(format_input(entry))
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {entry['gpt2_finetuned_response']}")

print("=" * 70)
print(f"Stopped at <|endoftext|>: {stopped_on_own}/{len(test_data)} "
      f"(the rest hit the {MAX_NEW_TOKENS}-token limit)")
print(f"Generated tokens per response: "
      f"mean {sum(response_lengths) / len(response_lengths):.1f}, "
      f"max {max(response_lengths)}")
print(f"Reference answer length for comparison: mean "
      f"{sum(len(tokenizer.encode(e['output'])) for e in test_data) / len(test_data):.1f} tokens")

## 13. Save

The checkpoint is ~500 MB, so downloading it through the browser is slow — mounting Drive and
copying it there is usually faster.

In [ ]:
torch.save(model.state_dict(), "gpt2-small124M-sft.pth")

with open("test-set-responses.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=4)

print("saved gpt2-small124M-sft.pth and test-set-responses.json")

In [ ]:
from google.colab import files

files.download("test-set-responses.json")
files.download("loss-plot.pdf")

# Big checkpoint — via Drive instead of the browser:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp gpt2-small124M-sft.pth /content/drive/MyDrive/

### Reloading the finetuned model later

```python
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("gpt2-small124M-sft.pth", map_location=device))
model.to(device).eval()
```